In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Install Dependencies & Setup Environment

In [ ]:
import os
import sys
import glob
import json
import re
import copy
import warnings
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageOps
from shutil import copyfile
from sklearn.impute import KNNImputer
from sklearn.model_selection import GroupShuffleSplit, StratifiedKFold

warnings.filterwarnings('ignore')

# Config Path and Constant

In [ ]:
BASE_INPUT = '/kaggle/input/datasets/akhyaruu/appendicitis'
BASE_OUTPUT = '/kaggle/working/Processed'

# Directory Setup
IMAGE_DIR = os.path.join(BASE_INPUT, 'US_Pictures_kaggle_ready')
TRAIN_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Train')
TEST_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Test')
PREPROC_IMAGE_DIR = os.path.join(BASE_OUTPUT, 'Preprocessed')
OUTPUT_DIR = os.path.join(BASE_OUTPUT, 'Dictionaries')

CLINICAL_DATA_FILE = os.path.join(BASE_INPUT, 'app_data.xlsx')

# Blacklist Dummy (jika tidak ada)
BLACKLIST_FILE = os.path.join(BASE_INPUT, 'blacklist.csv')
if not os.path.exists(BLACKLIST_FILE):
    BLACKLIST_FILE = '/kaggle/working/empty_blacklist.csv'
    with open(BLACKLIST_FILE, 'w') as f:
        f.write("image_name\n")

# Konfigurasi
PADDING_MODE = 'constant'
TARGET_LABEL = 'diagnosis'

TABULAR_FEATURES = ['Age', 'Sex', 'Height', 'Weight', 'BMI', 'Alvarado_Score', 'Paedriatic_Appendicitis_Score',
                    'Peritonitis', 'Migratory_Pain', 'Lower_Right_Abd_Pain', 'Contralateral_Rebound_Tenderness', 
                    'Coughing_Pain', 'Psoas_Sign', 'Nausea', 'Loss_of_Appetite', 'Body_Temperature', 'Dysuria', 
                    'Stool', 'WBC_Count', 'Neutrophil_Percentage', 'CRP', 'Ketones_in_Urine', 'RBC_in_Urine', 
                    'WBC_in_Urine', 'Appendix_on_US', 'Appendix_Diameter', 'Free_Fluids', 
                    'Appendix_Wall_Layers', 'Target_Sign', 'Perfusion', 'Perforation', 
                    'Surrounding_Tissue_Reaction', 'Pathological_Lymph_Nodes', 'Bowel_Wall_Thickening', 'Ileus', 
                    'Coprostasis', 'Meteorism', 'Enteritis', 'Appendicular_Abscess', 
                    'Conglomerate_of_Bowel_Loops', 'Gynecological_Findings']

REAL_VALUED = ['Age', 'Height', 'Weight', 'BMI', 'Body_Temperature', 'WBC_Count', 'Neutrophil_Percentage', 
               'CRP', 'Appendix_Diameter']

CONCEPTS = ['Appendix_on_US', 'Free_Fluids', 'Appendix_Wall_Layers', 'Target_Sign', 'Surrounding_Tissue_Reaction',
            'Pathological_Lymph_Nodes', 'Bowel_Wall_Thickening', 'Coprostasis', 'Meteorism', 'Enteritis', 
            'Appendix_Diameter', 'Perforation', 'Appendicular_Abscess', 'Conglomerate_of_Bowel_Loops', 
            'Gynecological_Findings']

# Buat direktori output
for d in [TRAIN_IMAGE_DIR, TEST_IMAGE_DIR, PREPROC_IMAGE_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Konfigurasi Path Selesai.")

# Split Data & Rename File

In [ ]:
def test_split(train_image_dir, test_image_dir):
    train_files = glob.glob(os.path.join(train_image_dir, '*'))
    test_files = glob.glob(os.path.join(test_image_dir, '*'))
    
    train_groups = np.array([int(re.split('_| |\\.', os.path.basename(f))[0]) for f in train_files])
    test_groups = np.array([int(re.split('_| |\\.', os.path.basename(f))[0]) for f in test_files])
    
    print(f"Train: {len(np.unique(train_groups))} patients, {len(train_groups)} images")
    print(f"Test: {len(np.unique(test_groups))} patients, {len(test_groups)} images")
    
    if len(np.intersect1d(train_groups, test_groups)) == 0:
        print("✅ Patients in train and test set don't overlap!")

# 1. Split dataset
print("Memisahkan Train dan Test...")
image_file_list = glob.glob(os.path.join(IMAGE_DIR, '*'))
image_names = [os.path.basename(f) for f in image_file_list]
groups = np.array([int(re.split('_| |\\.', name)[0]) for name in image_names])

gss = GroupShuffleSplit(n_splits=1, train_size=.9, random_state=42)
for idx1, idx2 in gss.split(image_names, groups=groups):
    train_idx, test_idx = idx1, idx2

for image_idx, image_name in enumerate(image_names):
    src_path = os.path.join(IMAGE_DIR, image_name)
    if image_idx in train_idx:
        copyfile(src_path, os.path.join(TRAIN_IMAGE_DIR, image_name))
    else:
        copyfile(src_path, os.path.join(TEST_IMAGE_DIR, image_name))

test_split(TRAIN_IMAGE_DIR, TEST_IMAGE_DIR)

# 2. Rename file aneh (spasi, umlaut, dll)
def rename_weird_files(folder):
   files = glob.glob(folder + '/*')
   for f in files:
      name = os.path.basename(f)
      if any(c in name for c in ['ä', 'Ä', 'ö', 'Ö', 'ü', 'Ü', 'ß', ' ']):
         try:
            img = Image.open(f)
            new_name = name.replace(' ', '_').replace('ä', 'ae').replace('Ä', 'Ae').replace('ö', 'oe').replace('Ö', 'Oe').replace('ü', 'ue').replace('Ü', 'Ue').replace('ß', 'ss')
            new_path = os.path.join(folder, new_name)
            img.save(new_path)
            os.remove(f)
         except Exception:
            pass

print("Membersihkan nama file...")
rename_weird_files(TRAIN_IMAGE_DIR)
rename_weird_files(TEST_IMAGE_DIR)

# Pembersihan Gambar (DeepFill & Cropping)

In [ ]:
def process_images(src_dir, out_dir, target_size=(224, 224)):
    print(f"Memproses (Inpainting + Crop) gambar dari {os.path.basename(src_dir)}...")
    os.makedirs(out_dir, exist_ok=True)
    files = glob.glob(os.path.join(src_dir, '*'))
    
    for f in files:
        filename = os.path.basename(f)
        img = cv2.imread(f)
        if img is None: continue
            
        # 1. Inpainting (DeepFill Replacement)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY) # Deteksi teks putih
        img_inpainted = cv2.inpaint(img, mask, 3, cv2.INPAINT_TELEA)
        
        # 2. Cropping & Padding
        img_rgb = cv2.cvtColor(img_inpainted, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        img_padded = ImageOps.pad(pil_img, target_size, color=(0, 0, 0))
        
        # Simpan
        img_padded.save(os.path.join(out_dir, filename))

# Jalankan proses
TRAIN_PROCESSED = os.path.join(PREPROC_IMAGE_DIR, 'cropped_train')
TEST_PROCESSED = os.path.join(PREPROC_IMAGE_DIR, 'cropped_test')

process_images(TRAIN_IMAGE_DIR, TRAIN_PROCESSED)
process_images(TEST_IMAGE_DIR, TEST_PROCESSED)

# Generate JSON & Imputasi

In [ ]:
def generate_json_mapping(image_dir, excel_file, output_filepath, target_label='Diagnosis'):
    print(f"Membuat mapping JSON untuk {os.path.basename(image_dir)}...")
    df = pd.read_excel(excel_file)
    
    # Cari nama kolom ID di Excel (Asumsi kolom pertama atau bernama 'ID')
    id_col = df.columns[0] 
    if 'Patient_ID' in df.columns: id_col = 'Patient_ID'
    
    available_images = glob.glob(os.path.join(image_dir, '*'))
    mapping = {}
    
    for img_path in available_images:
        filename = os.path.basename(img_path)
        try:
            patient_id = int(filename.split('.')[0]) 
        except ValueError:
            continue
            
        row = df[df[id_col] == patient_id]
            
        if not row.empty:
            # Konversi label target ke biner (1 atau 0)
            label_val = 1 if 'appendicitis' in str(row[target_label].values[0]).lower() else 0
            
            features = row[TABULAR_FEATURES].values[0].tolist()
            concepts = row[CONCEPTS].values[0].tolist()
            
            mapping[filename] = [filename, int(label_val), features, concepts]
            
    os.makedirs(os.path.dirname(output_filepath), exist_ok=True)
    with open(output_filepath, 'w') as f:
        json.dump(mapping, f)
    print(f"Selesai. {len(mapping)} gambar berhasil dimapping.")

# 1. Generate JSON
TRAIN_JSON = os.path.join(OUTPUT_DIR, TARGET_LABEL, 'app_data_train.json')
TEST_JSON = os.path.join(OUTPUT_DIR, TARGET_LABEL, 'app_data_test.json')

generate_json_mapping(TRAIN_PROCESSED, CLINICAL_DATA_FILE, TRAIN_JSON, 'Diagnosis')
generate_json_mapping(TEST_PROCESSED, CLINICAL_DATA_FILE, TEST_JSON, 'Diagnosis')

# 2. Imputasi Data Kosong (KNN)
def decompose_dict(f):
    keys, image_names, X, y = [], [], [], []
    for i in range(len(f)):
        keys.append(list(f)[i])
        image_names.append(list(f.values())[i][0])
        y.append(list(f.values())[i][1])
        X.append(list(f.values())[i][2])
    return keys, image_names, pd.DataFrame(X, columns=TABULAR_FEATURES), np.array(y)

def impute_final(keys, image_names, X, y, dataset_name, keys_test, image_names_test, X_test, y_test, test_dataset_name):
    print(f"Melakukan Imputasi data kosong untuk {dataset_name} & {test_dataset_name}...")
    mapping, mapping_test = {}, {}
    imputer = KNNImputer(n_neighbors=1)
    
    X = pd.DataFrame(imputer.fit_transform(X), columns=TABULAR_FEATURES)
    X_test = pd.DataFrame(imputer.transform(X_test), columns=TABULAR_FEATURES)
    X_norm, X_test_norm = copy.deepcopy(X), copy.deepcopy(X_test)
    
    for real_var in REAL_VALUED:
        mean, std = np.mean(X[real_var].values), np.std(X[real_var].values)
        X_norm[real_var] = (X[real_var]-mean)/std
        X_test_norm[real_var] = (X_test[real_var]-mean)/std        
    
    concepts, concepts_test = X_norm[CONCEPTS], X_test_norm[CONCEPTS]
    
    for idx, key in enumerate(keys):
        mapping[key] = [image_names[idx], int(y[idx]), X_norm.iloc[idx].values.tolist(), concepts.iloc[idx].values.tolist()]
    for idx, key in enumerate(keys_test):
        mapping_test[key] = [image_names_test[idx], int(y_test[idx]), X_test_norm.iloc[idx].values.tolist(), concepts_test.iloc[idx].values.tolist()]
    
    out_path = os.path.join(OUTPUT_DIR, TARGET_LABEL, 'imputed')
    os.makedirs(out_path, exist_ok=True)
    
    with open(os.path.join(out_path, dataset_name), 'w') as f:
        json.dump(mapping, f) 
    with open(os.path.join(out_path, test_dataset_name), 'w') as f:
        json.dump(mapping_test, f) 

# Load JSON lalu Imputasi
with open(TRAIN_JSON) as f:
    keys, image_names, X, y = decompose_dict(json.load(f))
with open(TEST_JSON) as f:
    keys_test, image_names_test, X_test, y_test = decompose_dict(json.load(f))

impute_final(keys, image_names, X, y, 'app_data_train_imputed.json', keys_test, image_names_test, X_test, y_test, 'app_data_test_imputed.json')

print("\n✅ SEMUA PROSES PREPROCESSING SELESAI!")
print(f"File JSON siap pakai ada di: {os.path.join(OUTPUT_DIR, TARGET_LABEL, 'imputed')}")